In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util, json, subprocess, sys, time
import numpy as np
import pandas as pd
from scipy.optimize import minimize

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
CASE_ROWS = [4923, 5192, 27897, 27901, 40099, 40114, 63014, 63018, 86891, 87483, 89898, 89906]

if importlib.util.find_spec("numpyro") is None:
    print("Installing NumPyro without changing Kaggle's JAX/CUDA packages...")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "numpyro==0.22.0"], check=True)
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from numpyro.infer.initialization import init_to_value
from numpyro.diagnostics import summary as numpyro_summary
import matplotlib.pyplot as plt

print("JAX backend:", jax.default_backend(), "| devices:", jax.devices(), "| NumPyro:", numpyro.__version__)
if jax.default_backend() != "gpu":
    raise RuntimeError("GPU preflight failed: set Kaggle Accelerator to GPU T4 x2, restart the session, then Run All.")

binary_sources = []
for mp in INPUT.rglob("simulation_metadata.json"):
    if mp.parent.name != "binary":
        continue
    try:
        meta = json.loads(mp.read_text(encoding="utf-8"))
    except Exception:
        continue
    if meta.get("family") == "binary" and int(meta.get("parameter_draws_unique", 0)) >= 100_000:
        binary_sources.append((meta.get("created_utc", ""), mp.parent))
if not binary_sources:
    raise FileNotFoundError("Attach Day 7 Cosmic swan Bayesian output with 100K binary simulations.")
_, BINARY_DIR = sorted(binary_sources, key=lambda x: x[0])[-1]
observations = np.load(BINARY_DIR / "observations.npy", mmap_mode="r")
parameters = pd.read_csv(BINARY_DIR / "parameters.csv")
time_days = np.asarray(np.load(BINARY_DIR / "time_days.npy"), dtype=float)
if observations.shape != (len(parameters), len(time_days)) or max(CASE_ROWS) >= len(parameters):
    raise ValueError("Binary observation, parameter, time-axis, or selected row dimensions do not match.")

score_paths = list(INPUT.rglob("day9_3_binary_alias_candidate_scores.csv"))
if not score_paths:
    raise FileNotFoundError("Attach Day 9.3 output with day9_3_binary_alias_candidate_scores.csv.")
score_path = max(score_paths, key=lambda p: p.stat().st_mtime)
candidate_df = pd.read_csv(score_path)
required = {"simulation_row", "bic", "fit_period_days", "phase", "duration_days", "noise_sigma"}
if not required.issubset(candidate_df.columns):
    raise ValueError(f"Day 9.3 score CSV is missing columns: {sorted(required - set(candidate_df.columns))}")
if not set(CASE_ROWS).issubset(set(candidate_df["simulation_row"].astype(int))):
    raise ValueError("Day 9.3 score CSV does not contain all 12 fixed stress-test rows.")

OUT = WORK / ("day9-7-12case-binary-nuts-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
OUT.mkdir(parents=True, exist_ok=False)
CHAINS, WARMUP, SAMPLES = 2, 500, 800
CHAIN_METHOD = "parallel" if len(jax.devices()) >= 2 else "sequential"
print("[PASS] Day 7 binary input:", BINARY_DIR)
print("[PASS] Day 9.3 alias scores:", score_path)
print("[PASS] Fixed cases:", CASE_ROWS, "| output:", OUT)
print("NUTS:", CHAINS, "chains x", WARMUP, "warmup +", SAMPLES, "draws; chain method:", CHAIN_METHOD)


In [ ]:
def smooth_clip_np(x, sharpness=500.0):
    return (np.logaddexp(0.0, sharpness*x) - np.logaddexp(0.0, sharpness*(x-1.0))) / sharpness

def predict_np(t, baseline, period, phase, depth, ratio, duration, ellip, reflection):
    orb = np.mod(t/period-phase+0.5, 1.0)-0.5
    sec = np.mod(orb, 1.0)-0.5
    half = max(float(duration)/2.0, 1e-6)
    ingress = max(half*0.18, 1e-6)
    primary = smooth_clip_np((half+ingress-np.abs(orb))/(2.0*ingress))
    secondary = smooth_clip_np((half+ingress-np.abs(sec))/(2.0*ingress))
    return baseline-depth*primary-depth*ratio*secondary + ellip*np.cos(4*np.pi*orb) + reflection*np.cos(2*np.pi*orb)

map_cases, map_parameters, map_plot_rows = [], [], []
for row in CASE_ROWS:
    started = time.time()
    matching = candidate_df[candidate_df["simulation_row"].astype(int) == int(row)].sort_values("bic")
    prior = matching.iloc[0]
    t = np.asarray(time_days, float)
    y = np.asarray(observations[int(row)], float)
    ok = np.isfinite(t) & np.isfinite(y)
    t, y = t[ok], y[ok]
    order = np.argsort(t)
    t, y = t[order], y[order]
    t = t-t.min()  # Preserve raw simulator flux; do not median-normalize.

    center = float(prior["fit_period_days"])
    window = max(0.15, 0.02*center)
    p_low, p_high = max(0.1, center-window), min(13.0, center+window)
    phase0 = float(np.mod(float(prior["phase"]), 1.0))
    duration0 = float(np.clip(float(prior["duration_days"])/center, 0.005, 0.12))
    noise0 = float(np.clip(float(prior["noise_sigma"]), 1.0001e-4, 0.02999))
    depth0 = float(np.clip((np.nanpercentile(y, 95)-np.nanpercentile(y, 5))/2, 0.0021, 0.4499))
    baseline0 = float(np.clip(np.nanpercentile(y, 90), 0.9501, 1.0499))
    bounds = [(0.95, 1.05), (p_low, p_high), (0.0, 1.0), (0.002, 0.45),
              (0.0, 1.0), (0.005, 0.12), (0.0, 0.02), (-0.01, 0.01),
              (np.log(1e-4), np.log(0.03))]

    def nll(theta):
        pred = predict_np(t, *theta[:8])
        log_noise = theta[8]
        return float(0.5*np.sum(((y-pred)/np.exp(log_noise))**2 + 2*log_noise + np.log(2*np.pi)))

    starts = []
    for shift in (0.0, 0.5):
        x = np.array([baseline0, center, (phase0+shift)%1.0, depth0, 0.5,
                      duration0, 0.005, 0.0, np.log(noise0)])
        starts.append(np.array([np.clip(v, lo+1e-9, hi-1e-9) for v, (lo, hi) in zip(x, bounds)]))
    fits = [minimize(nll, x, method="L-BFGS-B", bounds=bounds,
                     options={"maxiter": 1500, "ftol": 1e-10}) for x in starts]
    fit = min(fits, key=lambda z: z.fun if np.isfinite(z.fun) else np.inf)
    theta = np.asarray(fit.x, float)
    names = ["baseline_flux", "period_days", "phase", "depth_primary", "depth_ratio",
             "duration_fraction", "ellipsoidal_amplitude", "reflection_amplitude", "log_noise"]
    estimates = dict(zip(names, theta.tolist()))
    estimates["noise_sigma"] = float(np.exp(theta[8]))
    truth = parameters.iloc[int(row)]
    truth_map = {name: float(truth[name]) for name in
                 ["period_days", "phase", "depth_primary", "depth_ratio", "duration_fraction",
                  "ellipsoidal_amplitude", "reflection_amplitude", "noise_sigma"]}
    truth_map["baseline_flux"] = 1.0
    phase_error = float((estimates["phase"]-truth_map["phase"]+0.5)%1.0-0.5)
    period_tol = max(0.01, 0.002*truth_map["period_days"])
    row_summary = {"simulation_row": int(row), "optimizer_success": bool(fit.success),
        "starting_period_days": center, "map_period_days": float(theta[1]),
        "known_period_days_for_evaluation": truth_map["period_days"],
        "map_period_abs_error_days": abs(float(theta[1])-truth_map["period_days"]),
        "period_recovered_within_tolerance": bool(abs(float(theta[1])-truth_map["period_days"]) <= period_tol),
        "period_evaluation_tolerance_days": period_tol,
        "phase_abs_error_cycles": abs(phase_error), "map_noise_sigma": estimates["noise_sigma"],
        "seconds": round(time.time()-started, 1)}
    map_cases.append(row_summary)
    for name in ["baseline_flux", "period_days", "phase", "depth_primary", "depth_ratio",
                 "duration_fraction", "ellipsoidal_amplitude", "reflection_amplitude", "noise_sigma"]:
        value = estimates[name]
        target = truth_map[name]
        error = phase_error if name == "phase" else value-target
        map_parameters.append({"simulation_row": int(row), "parameter": name,
                               "map_estimate": value, "known_truth_for_evaluation": target,
                               "estimate_minus_truth": error, "absolute_error": abs(error)})
    map_plot_rows.append((int(row), t, y, predict_np(t, *theta[:8])))
    pd.DataFrame(map_cases).to_csv(OUT/"day9_7_map_case_summary.csv", index=False)
    pd.DataFrame(map_parameters).to_csv(OUT/"day9_7_map_parameter_recovery.csv", index=False)
    print(f"MAP row {row}: period={theta[1]:.6f} d; truth error={row_summary['map_period_abs_error_days']:.5g} d; {row_summary['seconds']} s", flush=True)

map_case_df = pd.DataFrame(map_cases)
map_parameter_df = pd.DataFrame(map_parameters)
map_df = pd.DataFrame([{"simulation_row": r["simulation_row"], "parameter": r["parameter"],
                        "map_estimate": r["map_estimate"]} for r in map_parameters])
print("MAP period recovery:", int(map_case_df["period_recovered_within_tolerance"].sum()), "/", len(map_case_df))


In [ ]:
def smooth_clip_jax(x, sharpness=500.0):
    return (jax.nn.softplus(sharpness*x)-jax.nn.softplus(sharpness*(x-1.0)))/sharpness

def predict_jax(t, period, phase, depth, ratio, duration, ellip, reflection, baseline):
    orb = jnp.mod(t/period-phase+0.5, 1.0)-0.5
    sec = jnp.mod(orb, 1.0)-0.5
    half = jnp.maximum(duration/2.0, 1e-6)
    ingress = jnp.maximum(half*0.18, 1e-6)
    primary = smooth_clip_jax((half+ingress-jnp.abs(orb))/(2.0*ingress))
    secondary = smooth_clip_jax((half+ingress-jnp.abs(sec))/(2.0*ingress))
    return baseline-depth*primary-depth*ratio*secondary + ellip*jnp.cos(4*jnp.pi*orb) + reflection*jnp.cos(2*jnp.pi*orb)

def scalar_metric(diagnostics, key):
    values = []
    for item in diagnostics.values():
        if key in item:
            arr = np.asarray(item[key])
            if arr.size == 1 and np.isfinite(arr).all():
                values.append(float(arr.reshape(-1)[0]))
    return values

nuts_cases, interval_rows, posterior_draws, plot_rows = [], [], {}, []
for row in CASE_ROWS:
    started = time.time()
    subset = map_df[map_df["simulation_row"].astype(int) == int(row)]
    map_values = subset.set_index("parameter")["map_estimate"].astype(float).to_dict()
    needed = {"baseline_flux", "period_days", "phase", "depth_primary", "depth_ratio",
              "duration_fraction", "ellipsoidal_amplitude", "reflection_amplitude", "noise_sigma"}
    if not needed.issubset(map_values):
        raise ValueError(f"Missing MAP seeds for row {row}: {sorted(needed-set(map_values))}")
    t = np.asarray(time_days, float)
    y = np.asarray(observations[int(row)], float)
    ok = np.isfinite(t) & np.isfinite(y)
    t, y = t[ok], y[ok]
    order = np.argsort(t)
    t, y = t[order], y[order]
    t = t-t.min()  # Same raw flux scale used by the simulator truths.
    period0 = float(map_values["period_days"])
    period_width = max(0.15, 0.02*period0)
    p_low, p_high = max(0.1, period0-period_width), min(13.0, period0+period_width)
    phase0 = float(np.mod(map_values["phase"], 1.0))
    observed_j, time_j = jnp.asarray(y), jnp.asarray(t)

    def model():
        baseline = numpyro.sample("baseline_flux", dist.Normal(1.0, 0.02))
        period = numpyro.sample("period_days", dist.Uniform(p_low, p_high))
        phase_offset = numpyro.sample("phase_offset", dist.Normal(0.0, 0.025))
        depth = numpyro.sample("depth_primary", dist.Uniform(0.002, 0.45))
        ratio = numpyro.sample("depth_ratio", dist.Uniform(0.0, 1.0))
        duration = numpyro.sample("duration_fraction", dist.Uniform(0.005, 0.12))
        ellip = numpyro.sample("ellipsoidal_amplitude", dist.Uniform(0.0, 0.02))
        reflection = numpyro.sample("reflection_amplitude", dist.Uniform(-0.01, 0.01))
        log_noise = numpyro.sample("log_noise", dist.Uniform(np.log(1e-4), np.log(0.03)))
        phase = jnp.mod(phase0+phase_offset, 1.0)
        pred = predict_jax(time_j, period, phase, depth, ratio, duration, ellip, reflection, baseline)
        numpyro.sample("flux", dist.Normal(pred, jnp.exp(log_noise)), obs=observed_j)

    init = {"baseline_flux": float(np.clip(map_values["baseline_flux"], 0.9501, 1.0499)),
        "period_days": float(np.clip(period0, p_low+1e-6, p_high-1e-6)), "phase_offset": 0.0,
        "depth_primary": float(np.clip(map_values["depth_primary"], 0.002001, 0.449999)),
        "depth_ratio": float(np.clip(map_values["depth_ratio"], 1e-5, 0.99999)),
        "duration_fraction": float(np.clip(map_values["duration_fraction"], 0.005001, 0.119999)),
        "ellipsoidal_amplitude": float(np.clip(map_values["ellipsoidal_amplitude"], 1e-5, 0.01999)),
        "reflection_amplitude": float(np.clip(map_values["reflection_amplitude"], -0.00999, 0.00999)),
        "log_noise": float(np.log(np.clip(map_values["noise_sigma"], 1.0001e-4, 0.02999)))}
    kernel = NUTS(model, init_strategy=init_to_value(values=init), target_accept_prob=0.95,
                  max_tree_depth=10, dense_mass=True)
    mcmc = MCMC(kernel, num_warmup=WARMUP, num_samples=SAMPLES, num_chains=CHAINS,
                chain_method=CHAIN_METHOD, progress_bar=True)
    try:
        mcmc.run(jax.random.PRNGKey(95000+int(row)))
        grouped = mcmc.get_samples(group_by_chain=True)
        flat = mcmc.get_samples(group_by_chain=False)
        diagnostics = numpyro_summary(grouped, group_by_chain=True)
        rhats, esses = scalar_metric(diagnostics, "r_hat"), scalar_metric(diagnostics, "n_eff")
        divergences = int(np.asarray(mcmc.get_extra_fields(group_by_chain=True)["diverging"]).sum())
        max_rhat = max(rhats) if rhats else float("nan")
        min_ess = min(esses) if esses else float("nan")
        status = "screen_pass" if np.isfinite(max_rhat) and np.isfinite(min_ess) and max_rhat <= 1.05 and min_ess >= 100 and divergences == 0 else "needs_review"
        transformed = {k: np.asarray(v) for k, v in flat.items()}
        transformed["phase"] = (phase0+transformed["phase_offset"])%1.0
        transformed["noise_sigma"] = np.exp(transformed["log_noise"])
        np.savez_compressed(OUT/f"binary_row_{row}_posterior.npz", **transformed)
        truth = parameters.iloc[int(row)]
        truth_map = {name: float(truth[name]) for name in
            ["period_days", "phase", "depth_primary", "depth_ratio", "duration_fraction",
             "ellipsoidal_amplitude", "reflection_amplitude", "noise_sigma"]}
        truth_map["baseline_flux"] = 1.0
        for name in ["baseline_flux", "period_days", "phase_offset", "depth_primary", "depth_ratio",
                     "duration_fraction", "ellipsoidal_amplitude", "reflection_amplitude", "noise_sigma"]:
            samples = transformed[name] if name in transformed else transformed["noise_sigma"]
            q05, median, q95 = np.quantile(samples, [0.05, 0.5, 0.95])
            phase_truth = (truth_map["phase"]-phase0+0.5)%1.0-0.5
            target = phase_truth if name == "phase_offset" else truth_map[name]
            interval_rows.append({"simulation_row": int(row), "parameter": name, "q05": float(q05),
                "median": float(median), "q95": float(q95), "known_truth_or_phase_offset": float(target),
                "truth_inside_90pct_interval": bool(q05 <= target <= q95)})
        med = {k: float(np.median(v)) for k, v in transformed.items()}
        fitted_phase = float((phase0+med["phase_offset"])%1.0)
        pred = np.asarray(predict_jax(time_j, med["period_days"], fitted_phase, med["depth_primary"],
            med["depth_ratio"], med["duration_fraction"], med["ellipsoidal_amplitude"],
            med["reflection_amplitude"], med["baseline_flux"]), float)
        plot_rows.append((int(row), t, y, pred))
        nuts_cases.append({"simulation_row": int(row), "status": status, "period_low": p_low,
            "period_high": p_high, "period_median": float(np.median(transformed["period_days"])),
            "max_r_hat": max_rhat, "min_effective_sample_size": min_ess,
            "divergences": divergences, "seconds": round(time.time()-started, 1)})
        posterior_draws[int(row)] = transformed
    except Exception as exc:
        nuts_cases.append({"simulation_row": int(row), "status": "failed",
            "error": f"{type(exc).__name__}: {exc}", "period_low": p_low,
            "period_high": p_high, "seconds": round(time.time()-started, 1)})
    pd.DataFrame(nuts_cases).to_csv(OUT/"day9_7_nuts_case_diagnostics.csv", index=False)
    pd.DataFrame(interval_rows).to_csv(OUT/"day9_7_posterior_recovery_intervals.csv", index=False)
    print(f"NUTS row {row}: {nuts_cases[-1]}", flush=True)


In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 14), constrained_layout=True)
for ax, (row, t, y, pred) in zip(axes.flat, plot_rows):
    ax.scatter(t, y, s=5, alpha=0.35, label="synthetic observations")
    ax.plot(t, pred, color="crimson", lw=0.9, label="posterior-median fit")
    ax.set_title(f"Row {row} | conditional binary fit")
    ax.set_xlabel("Time (days)")
    ax.set_ylabel("Flux")
    ax.legend(fontsize=7)
for ax in axes.flat[len(plot_rows):]:
    ax.set_visible(False)
fit_plot = OUT/"day9_7_posterior_median_fits.png"
fig.savefig(fit_plot, dpi=140)
plt.show()

map_case_df = pd.DataFrame(map_cases)
nuts_case_df = pd.DataFrame(nuts_cases)
interval_df = pd.DataFrame(interval_rows)
coverage = {}
if not interval_df.empty:
    coverage = {name: {"n": int(len(group)), "covered": int(group["truth_inside_90pct_interval"].sum()),
                       "fraction": float(group["truth_inside_90pct_interval"].mean())}
                for name, group in interval_df.groupby("parameter")}
summary = {"created_utc": datetime.now(timezone.utc).isoformat(),
    "jax_backend": jax.default_backend(), "jax_devices": [str(d) for d in jax.devices()],
    "day7_binary_input": str(BINARY_DIR), "day9_3_alias_input": str(score_path),
    "scope": "12 fixed Day 9.3 synthetic alias-stress cases; raw-scale MAP seeds and NUTS posteriors conditional on one BIC-selected local period mode.",
    "configuration": {"cases": CASE_ROWS, "chains": CHAINS, "warmup_per_chain": WARMUP,
        "samples_per_chain": SAMPLES, "target_accept_prob": 0.95,
        "dense_mass": True, "chain_method": CHAIN_METHOD},
    "map_period_recovered_count_evaluation_only": int(map_case_df["period_recovered_within_tolerance"].sum()),
    "map_period_case_count": int(len(map_case_df)),
    "nuts_screen_pass_count": int(nuts_case_df.get("status", pd.Series(dtype=str)).eq("screen_pass").sum()),
    "nuts_needs_review_or_failed_count": int(nuts_case_df.get("status", pd.Series(dtype=str)).isin(["needs_review", "failed"]).sum()),
    "descriptive_90pct_interval_coverage": coverage,
    "outputs": ["day9_7_map_case_summary.csv", "day9_7_map_parameter_recovery.csv",
        "day9_7_nuts_case_diagnostics.csv", "day9_7_posterior_recovery_intervals.csv",
        "binary_row_*_posterior.npz", "day9_7_posterior_median_fits.png"],
    "limits": ["12 cases are a fixed stress set, not an IID sample; coverage is descriptive and does not establish calibrated 90% coverage.",
        "NUTS posteriors condition on one BIC-selected local mode; global aliases are not marginalized.",
        "MAP and NUTS period errors are evaluated against known synthetic truth; truth is not used for mode selection.",
        "Sampler screen-pass does not establish model correctness or real-star validity.",
        "These are synthetic simulations, not posterior claims for real TESS candidates."]}
(OUT/"day9_7_12case_binary_nuts_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("MAP case results:")
print(map_case_df.to_string(index=False))
print("NUTS diagnostics:")
print(nuts_case_df.to_string(index=False))
print("90% interval recovery (descriptive):")
print(json.dumps(coverage, indent=2))
print(json.dumps(summary, indent=2))
print("Outputs:", OUT)